In [ ]:
import bql
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots

bq = bql.Service()


In [ ]:
# ==========================================================================
# CONSTANTS
# ==========================================================================

INDEX_MAP = {
    "SPX": "SPX Index",
    "NDX": "NDX Index",
    "INDU": "INDU Index",
    "RTY": "RTY Index",
}

# BQL implied_volatility EXPIRY values (must match exactly)
EXPIRY_OPTIONS = ["30D", "60D", "90D", "180D", "360D", "540D", "720D"]

# BQL implied_volatility DELTA values
DELTA_OPTIONS = ["10", "25", "40", "50", "60", "75", "90"]

# BQL implied_volatility PCT_MONEYNESS values
MONEYNESS_OPTIONS = ["80", "90", "95", "97", "97.5", "100", "102", "102.5", "105", "110", "120"]

# BQL implied_volatility SIGMA values
SIGMA_OPTIONS = ["NEG_1_5", "NEG_1_0", "NEG_0_5", "ATM", "POS_0_5", "POS_1_0", "POS_1_5"]

VIX_FUTURES_TICKERS = [
    "UX1 Index", "UX2 Index", "UX3 Index", "UX4 Index",
    "UX5 Index", "UX6 Index", "UX7 Index", "UX8 Index",
]

COLORS = {
    "iv": "#6CA6CD",
    "rv": "#FF8C69",
    "ratio": "#A580D0",
    "fwd": "#60D394",
    "fwd_factor": "#FFD166",
    "contango": "#60D394",
    "backwardation": "#FF6B6B",
    "flat": "#C0C0C0",
    "heatmap_low": "#2D8B46",
    "heatmap_mid": "#FFFFBF",
    "heatmap_high": "#D73027",
}

PLOTLY_LAYOUT = dict(
    template="plotly_dark",
    paper_bgcolor="#1E1E2F",
    plot_bgcolor="#1E1E2F",
    font=dict(family="Consolas, monospace", size=12, color="#E0E0E0"),
    margin=dict(l=60, r=30, t=50, b=50),
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)


In [ ]:
# ==========================================================================
# HELPERS
# ==========================================================================

def safe_bql(request, label="BQL"):
    try:
        return bq.execute(request)
    except Exception as e:
        print(f"[ERROR] {label}: {e}")
        return None

def resp_to_df(response, idx=0):
    if response is None:
        return pd.DataFrame()
    try:
        return response[idx].df()
    except Exception as e:
        print(f"[WARN] resp_to_df: {e}")
        return pd.DataFrame()

def multi_resp_to_dfs(response):
    if response is None:
        return []
    dfs = []
    for item in response:
        try:
            dfs.append(item.df())
        except Exception:
            dfs.append(pd.DataFrame())
    return dfs

def find_cols(df):
    """Return (date_col, value_col) from a BQL DataFrame."""
    cols = df.columns.tolist()
    date_c = next((c for c in cols if "DATE" in c.upper()), None)
    val_c = next((c for c in cols if c != date_c and "ID" not in c.upper()
                   and "CURRENCY" not in c.upper()), None)
    return date_c, val_c

def styled_fig(title="", height=420, rows=1, cols=1, shared_xaxes=True,
               subplot_titles=None, vertical_spacing=0.08):
    if rows > 1 or cols > 1:
        fig = make_subplots(rows=rows, cols=cols, shared_xaxes=shared_xaxes,
                            subplot_titles=subplot_titles,
                            vertical_spacing=vertical_spacing)
    else:
        fig = go.Figure()
    fig.update_layout(title=title, height=height, **PLOTLY_LAYOUT)
    return fig

def status_msg(text, color="#8899AA"):
    return HTML(f"<p style='color:{color}; font-family:Consolas,monospace;'>{text}</p>")

def error_msg(text):
    return status_msg(text, "#FF6B6B")



In [ ]:
# ==========================================================================
# TAB 1 — VIX TERM STRUCTURE REGIME
# ==========================================================================

def build_tab_vix_term_structure():
    out = widgets.Output()
    lookback_dd = widgets.Dropdown(options=["6M", "1Y", "2Y", "5Y"], value="1Y",
                                    description="Lookback:", style={"description_width": "80px"})
    refresh_btn = widgets.Button(description="Refresh", button_style="primary",
                                  layout=widgets.Layout(width="100px"))

    def _refresh(b=None):
        with out:
            clear_output(wait=True)
            display(status_msg("Loading VIX term structure…"))

            lb = lookback_dd.value
            tickers = ["VIX Index"] + VIX_FUTURES_TICKERS

            # Current prices — use list syntax
            req_px = bql.Request(tickers, {"px": bq.data.px_last()})
            resp = safe_bql(req_px, "VIX current")

            if resp is None:
                clear_output(wait=True)
                display(error_msg("Failed to fetch VIX data."))
                return

            df_px = resp_to_df(resp, 0)

            # Extract prices in order
            labels = ["VIX Spot"] + [f"UX{i}" for i in range(1, 9)]
            months = list(range(0, 9))
            prices = []
            for t in tickers:
                row = df_px[df_px.index == t]
                if not row.empty:
                    # Find the value column (not DATE, not CURRENCY)
                    val_cols = [c for c in row.columns if "DATE" not in c.upper()
                                and "CURRENCY" not in c.upper()]
                    if val_cols:
                        prices.append(float(row[val_cols[0]].iloc[0]))
                    else:
                        prices.append(np.nan)
                else:
                    prices.append(np.nan)

            # Historical VIX + UX1
            req_hist = bql.Request(
                ["VIX Index", "UX1 Index"],
                {"px": bq.data.px_last(dates=bq.func.range(f"-{lb}", "0D"))},
            )
            resp_hist = safe_bql(req_hist, "VIX history")

            clear_output(wait=True)

            # CHART 1: Current curve
            fig1 = styled_fig("VIX Futures Term Structure (Current)", height=320)
            valid_prices = [p for p in prices if not np.isnan(p)]
            curve_color = COLORS["contango"] if len(valid_prices) >= 2 and valid_prices[1] > valid_prices[0] else COLORS["backwardation"]
            fig1.add_trace(go.Scatter(
                x=months, y=prices, mode="lines+markers",
                marker=dict(size=10, color=curve_color),
                line=dict(color=curve_color, width=2.5),
                text=labels, hovertemplate="%{text}: %{y:.2f}<extra></extra>",
            ))
            fig1.update_xaxes(title_text="Contract", tickvals=months, ticktext=labels)
            fig1.update_yaxes(title_text="Level")

            regime = "CONTANGO" if curve_color == COLORS["contango"] else "BACKWARDATION"
            fig1.add_annotation(
                x=0.5, y=1.10, xref="paper", yref="paper",
                text=f"Regime: <b>{regime}</b>",
                showarrow=False, font=dict(size=14, color=curve_color),
            )
            display(go.FigureWidget(fig1))

            # CHART 2: Historical spread
            if resp_hist is not None:
                df_h = resp_to_df(resp_hist, 0)
                if not df_h.empty:
                    df_h = df_h.reset_index()
                    # BQL returns multi-ticker data with ID column
                    # Pivot: rows=dates, cols=tickers
                    dc, vc = find_cols(df_h)
                    if dc and vc:
                        vix_data = df_h[df_h["ID"] == "VIX Index"][[dc, vc]].rename(
                            columns={dc: "date", vc: "vix"}).dropna()
                        ux1_data = df_h[df_h["ID"] == "UX1 Index"][[dc, vc]].rename(
                            columns={dc: "date", vc: "ux1"}).dropna()

                        merged = pd.merge(vix_data, ux1_data, on="date", how="inner").sort_values("date")
                        merged["spread"] = merged["ux1"] - merged["vix"]
                        merged["regime_label"] = merged["spread"].apply(
                            lambda s: "Contango" if s > 0.5 else ("Backwardation" if s < -0.5 else "Flat"))

                        fig2 = styled_fig("VIX Spot vs Front-Month Future", height=460,
                                          rows=2, cols=1,
                                          subplot_titles=["VIX vs UX1", "UX1 − VIX Spread"],
                                          vertical_spacing=0.12)
                        fig2.add_trace(go.Scatter(x=merged["date"], y=merged["vix"], name="VIX Spot",
                                                   line=dict(color=COLORS["iv"], width=1.5)), row=1, col=1)
                        fig2.add_trace(go.Scatter(x=merged["date"], y=merged["ux1"], name="UX1",
                                                   line=dict(color=COLORS["rv"], width=1.5)), row=1, col=1)
                        fig2.add_trace(go.Bar(
                            x=merged["date"], y=merged["spread"], name="Spread",
                            marker_color=[COLORS["contango"] if s > 0 else COLORS["backwardation"]
                                          for s in merged["spread"]],
                            opacity=0.7), row=2, col=1)
                        fig2.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.4, row=2, col=1)
                        fig2.update_yaxes(title_text="Level", row=1, col=1)
                        fig2.update_yaxes(title_text="Spread", row=2, col=1)
                        display(go.FigureWidget(fig2))

                        # Stats
                        c_pct = (merged["regime_label"] == "Contango").mean() * 100
                        b_pct = (merged["regime_label"] == "Backwardation").mean() * 100
                        f_pct = (merged["regime_label"] == "Flat").mean() * 100
                        display(HTML(
                            f"<div style='color:#E0E0E0; font-family:Consolas,monospace; padding:8px; "
                            f"background:#1E1E2F; border-radius:4px; margin-top:8px;'>"
                            f"<b>Regime Breakdown ({lb}):</b>&nbsp;&nbsp;"
                            f"<span style='color:{COLORS['contango']}'>▮ Contango {c_pct:.1f}%</span>&nbsp;&nbsp;·&nbsp;&nbsp;"
                            f"<span style='color:{COLORS['backwardation']}'>▮ Backwardation {b_pct:.1f}%</span>&nbsp;&nbsp;·&nbsp;&nbsp;"
                            f"<span style='color:{COLORS['flat']}'>▮ Flat {f_pct:.1f}%</span></div>"))

    refresh_btn.on_click(_refresh)
    tab = widgets.VBox([widgets.HBox([lookback_dd, refresh_btn]), out])
    _refresh()
    return tab


In [ ]:
# ==========================================================================
# TAB 2 — RV vs IV ANALYSIS
# ==========================================================================

def build_tab_rv_iv():
    out = widgets.Output()
    idx_dd = widgets.Dropdown(options=list(INDEX_MAP.keys()), value="SPX",
                               description="Index:", style={"description_width": "80px"})
    term_dd = widgets.Dropdown(options=EXPIRY_OPTIONS, value="30D",
                                description="Expiry:", style={"description_width": "80px"})
    lb_dd = widgets.Dropdown(options=["6M", "1Y", "2Y", "5Y"], value="1Y",
                              description="Lookback:", style={"description_width": "80px"})
    refresh_btn = widgets.Button(description="Refresh", button_style="primary",
                                  layout=widgets.Layout(width="100px"))

    def _refresh(b=None):
        with out:
            clear_output(wait=True)
            idx = idx_dd.value
            ticker = INDEX_MAP[idx]
            expiry = term_dd.value
            lb = lb_dd.value

            display(status_msg(f"Loading {idx} {expiry} RV vs IV ({lb} lookback)…"))

            date_range = bq.func.range(f"-{lb}", "0D")

            # IV: ATM implied vol at this expiry
            iv_field = bq.data.implied_volatility(
                dates=date_range,
                expiry=expiry,
                delta="50",
                put_call="CALL",
            )

            # RV: compute from returns
            # BQL doesn't have realized_vol directly — compute from px_last
            # Use std of log returns over the period window
            term_days = int(expiry.replace("D", ""))
            rv_field = bq.func.std(
                bq.func.ln(bq.data.px_last(dates=date_range, per="D", fill="PREV")
                            / bq.data.px_last(dates=date_range, per="D", fill="PREV").shift(1)),
                term_days
            ) * bq.func.sqrt(252) * 100

            # Try IV first
            req_iv = bql.Request(ticker, {"IV": iv_field})
            resp_iv = safe_bql(req_iv, f"{idx} IV")

            if resp_iv is None:
                clear_output(wait=True)
                display(error_msg("IV query failed."))
                return

            df_iv = resp_to_df(resp_iv).reset_index()
            dc_iv, vc_iv = find_cols(df_iv)

            if not dc_iv or not vc_iv:
                clear_output(wait=True)
                display(error_msg("Could not parse IV data."))
                return

            iv_data = df_iv.rename(columns={dc_iv: "date", vc_iv: "iv"}).dropna(subset=["iv"])

            # Scale if in decimal
            if iv_data["iv"].median() < 1:
                iv_data["iv"] *= 100

            # For RV, try the computed field; if it fails, fall back to manual calc
            req_rv = bql.Request(ticker, {"RV": rv_field})
            resp_rv = safe_bql(req_rv, f"{idx} RV")

            rv_data = pd.DataFrame()
            if resp_rv is not None:
                df_rv = resp_to_df(resp_rv).reset_index()
                dc_rv, vc_rv = find_cols(df_rv)
                if dc_rv and vc_rv:
                    rv_data = df_rv.rename(columns={dc_rv: "date", vc_rv: "rv"}).dropna(subset=["rv"])

            # If computed RV failed, try manual approach
            if rv_data.empty:
                display(status_msg("Computing RV from price history…"))
                req_px = bql.Request(ticker, {
                    "px": bq.data.px_last(dates=date_range, per="D", fill="PREV")
                })
                resp_px = safe_bql(req_px, f"{idx} prices for RV")
                if resp_px is not None:
                    df_px = resp_to_df(resp_px).reset_index()
                    dc_px, vc_px = find_cols(df_px)
                    if dc_px and vc_px:
                        px = df_px.rename(columns={dc_px: "date", vc_px: "px"}).dropna().sort_values("date")
                        px["log_ret"] = np.log(px["px"] / px["px"].shift(1))
                        px["rv"] = px["log_ret"].rolling(window=term_days).std() * np.sqrt(252) * 100
                        rv_data = px[["date", "rv"]].dropna()

            if rv_data.empty:
                clear_output(wait=True)
                display(error_msg("Could not compute RV. Showing IV only."))
                # Show IV-only chart
                fig = styled_fig(f"{idx} — {expiry} Implied Volatility", height=400)
                fig.add_trace(go.Scatter(x=iv_data["date"], y=iv_data["iv"], name="IV",
                                          line=dict(color=COLORS["iv"], width=1.5)))
                fig.update_yaxes(title_text="IV (%)")
                display(go.FigureWidget(fig))
                return

            merged = pd.merge(iv_data[["date", "iv"]], rv_data[["date", "rv"]],
                              on="date", how="inner").sort_values("date")
            merged["ratio"] = merged["iv"] / merged["rv"]

            clear_output(wait=True)

            fig = styled_fig(
                f"{idx} — {expiry} IV vs RV", height=600, rows=2, cols=1,
                subplot_titles=[f"{expiry} Implied Vol vs Realized Vol", "IV / RV Ratio"],
                vertical_spacing=0.10)

            fig.add_trace(go.Scatter(x=merged["date"], y=merged["iv"], name="IV",
                                      line=dict(color=COLORS["iv"], width=1.5)), row=1, col=1)
            fig.add_trace(go.Scatter(x=merged["date"], y=merged["rv"], name="RV",
                                      line=dict(color=COLORS["rv"], width=1.5)), row=1, col=1)
            fig.update_yaxes(title_text="Vol (%)", row=1, col=1)

            fig.add_trace(go.Scatter(x=merged["date"], y=merged["ratio"], name="IV/RV",
                                      line=dict(color=COLORS["ratio"], width=1.5),
                                      fill="tozeroy", fillcolor="rgba(165,128,208,0.12)"), row=2, col=1)
            fig.add_hline(y=1.0, line_dash="dot", line_color="white", opacity=0.5, row=2, col=1)
            fig.update_yaxes(title_text="Ratio", row=2, col=1)

            # Stats
            if not merged.empty:
                c_iv = merged["iv"].iloc[-1]
                c_rv = merged["rv"].iloc[-1]
                c_r = merged["ratio"].iloc[-1]
                a_r = merged["ratio"].mean()
                fig.add_annotation(
                    x=0.01, y=0.98, xref="paper", yref="paper",
                    text=f"Latest → IV: {c_iv:.1f}  RV: {c_rv:.1f}  Ratio: {c_r:.2f}  (Avg: {a_r:.2f})",
                    showarrow=False, font=dict(size=11, color="#E0E0E0"),
                    bgcolor="rgba(30,30,47,0.8)", borderpad=4)

            display(go.FigureWidget(fig))

    refresh_btn.on_click(_refresh)
    tab = widgets.VBox([widgets.HBox([idx_dd, term_dd, lb_dd, refresh_btn]), out])
    _refresh()
    return tab


In [ ]:
# ==========================================================================
# TAB 3 — FORWARD VOL
# ==========================================================================

def build_tab_forward_vol():
    out = widgets.Output()
    idx_dd = widgets.Dropdown(options=list(INDEX_MAP.keys()), value="SPX",
                               description="Index:", style={"description_width": "80px"})
    t1_dd = widgets.Dropdown(options=EXPIRY_OPTIONS, value="30D",
                              description="Near:", style={"description_width": "80px"})
    t2_dd = widgets.Dropdown(options=EXPIRY_OPTIONS, value="90D",
                              description="Far:", style={"description_width": "80px"})
    strike_dd = widgets.Dropdown(
        options=["ATM (Δ50)"] + [f"{m}% Moneyness" for m in MONEYNESS_OPTIONS],
        value="ATM (Δ50)", description="Strike:", style={"description_width": "80px"})
    lb_dd = widgets.Dropdown(options=["6M", "1Y", "2Y", "5Y"], value="1Y",
                              description="Lookback:", style={"description_width": "80px"})
    refresh_btn = widgets.Button(description="Refresh", button_style="primary",
                                  layout=widgets.Layout(width="100px"))

    def _refresh(b=None):
        with out:
            clear_output(wait=True)
            t1 = t1_dd.value
            t2 = t2_dd.value
            t1_days = int(t1.replace("D", ""))
            t2_days = int(t2.replace("D", ""))

            if t1_days >= t2_days:
                display(status_msg("⚠ Near term must be shorter than far term.", "#FFD166"))
                return

            idx = idx_dd.value
            ticker = INDEX_MAP[idx]
            lb = lb_dd.value
            strike_sel = strike_dd.value
            date_range = bq.func.range(f"-{lb}", "0D")

            display(status_msg(f"Loading {idx} forward vol ({t1} → {t2})…"))

            # Build IV kwargs based on strike selection
            if strike_sel == "ATM (Δ50)":
                iv_near = bq.data.implied_volatility(dates=date_range, expiry=t1,
                                                      delta="50", put_call="CALL")
                iv_far = bq.data.implied_volatility(dates=date_range, expiry=t2,
                                                     delta="50", put_call="CALL")
            else:
                moneyness = strike_sel.split("%")[0]
                iv_near = bq.data.implied_volatility(dates=date_range, expiry=t1,
                                                      pct_moneyness=moneyness)
                iv_far = bq.data.implied_volatility(dates=date_range, expiry=t2,
                                                     pct_moneyness=moneyness)

            req = bql.Request(ticker, {"IV_NEAR": iv_near, "IV_FAR": iv_far})
            resp = safe_bql(req, "Forward vol")
            if resp is None:
                clear_output(wait=True)
                display(error_msg("Query failed."))
                return

            dfs = multi_resp_to_dfs(resp)
            if len(dfs) < 2:
                clear_output(wait=True)
                display(error_msg("Insufficient data."))
                return

            df_n = dfs[0].reset_index()
            df_f = dfs[1].reset_index()
            dc_n, vc_n = find_cols(df_n)
            dc_f, vc_f = find_cols(df_f)

            if not all([dc_n, vc_n, dc_f, vc_f]):
                clear_output(wait=True)
                display(error_msg("Could not parse columns."))
                return

            near = df_n.rename(columns={dc_n: "date", vc_n: "iv_near"}).dropna(subset=["iv_near"])
            far = df_f.rename(columns={dc_f: "date", vc_f: "iv_far"}).dropna(subset=["iv_far"])
            merged = pd.merge(near[["date", "iv_near"]], far[["date", "iv_far"]],
                              on="date", how="inner").sort_values("date")

            if merged["iv_near"].median() < 1:
                merged["iv_near"] *= 100
            if merged["iv_far"].median() < 1:
                merged["iv_far"] *= 100

            T1 = t1_days / 365.0
            T2 = t2_days / 365.0
            var_near = (merged["iv_near"] ** 2) * T1
            var_far = (merged["iv_far"] ** 2) * T2
            fwd_var = (var_far - var_near) / (T2 - T1)
            merged["fwd_vol"] = np.where(fwd_var > 0, np.sqrt(fwd_var), np.nan)

            clear_output(wait=True)

            fig = styled_fig(
                f"{idx} {strike_sel} — Forward Vol ({t1} → {t2})",
                height=520, rows=2, cols=1,
                subplot_titles=[f"Spot Vols: {t1} & {t2}", f"Forward Vol ({t1}→{t2})"],
                vertical_spacing=0.12)

            fig.add_trace(go.Scatter(x=merged["date"], y=merged["iv_near"], name=f"{t1} IV",
                                      line=dict(color=COLORS["iv"], width=1.5)), row=1, col=1)
            fig.add_trace(go.Scatter(x=merged["date"], y=merged["iv_far"], name=f"{t2} IV",
                                      line=dict(color=COLORS["rv"], width=1.5)), row=1, col=1)
            fig.update_yaxes(title_text="IV (%)", row=1, col=1)

            fig.add_trace(go.Scatter(x=merged["date"], y=merged["fwd_vol"], name="Fwd Vol",
                                      line=dict(color=COLORS["fwd"], width=2),
                                      fill="tozeroy", fillcolor="rgba(96,211,148,0.10)"), row=2, col=1)
            fig.update_yaxes(title_text="Fwd Vol (%)", row=2, col=1)

            display(go.FigureWidget(fig))

    refresh_btn.on_click(_refresh)
    tab = widgets.VBox([widgets.HBox([idx_dd, t1_dd, t2_dd, strike_dd, lb_dd, refresh_btn]), out])
    _refresh()
    return tab


In [ ]:
# ==========================================================================
# TAB 4 — FORWARD FACTOR
# ==========================================================================

def build_tab_forward_factor():
    out = widgets.Output()
    idx_dd = widgets.Dropdown(options=list(INDEX_MAP.keys()), value="SPX",
                               description="Index:", style={"description_width": "80px"})
    t1_dd = widgets.Dropdown(options=EXPIRY_OPTIONS, value="30D",
                              description="Near:", style={"description_width": "80px"})
    t2_dd = widgets.Dropdown(options=EXPIRY_OPTIONS, value="60D",
                              description="Far:", style={"description_width": "80px"})
    strike_dd = widgets.Dropdown(
        options=["ATM (Δ50)"] + [f"{m}% Moneyness" for m in MONEYNESS_OPTIONS],
        value="ATM (Δ50)", description="Strike:", style={"description_width": "80px"})
    lb_dd = widgets.Dropdown(options=["6M", "1Y", "2Y", "5Y"], value="1Y",
                              description="Lookback:", style={"description_width": "80px"})
    refresh_btn = widgets.Button(description="Refresh", button_style="primary",
                                  layout=widgets.Layout(width="100px"))

    def _refresh(b=None):
        with out:
            clear_output(wait=True)
            t1 = t1_dd.value
            t2 = t2_dd.value
            t1_days = int(t1.replace("D", ""))
            t2_days = int(t2.replace("D", ""))

            if t1_days >= t2_days:
                display(status_msg("⚠ Near term must be shorter than far term.", "#FFD166"))
                return

            idx = idx_dd.value
            ticker = INDEX_MAP[idx]
            lb = lb_dd.value
            strike_sel = strike_dd.value
            date_range = bq.func.range(f"-{lb}", "0D")

            display(status_msg(f"Loading {idx} forward factor ({t1} / {t1}→{t2})…"))

            if strike_sel == "ATM (Δ50)":
                iv_near = bq.data.implied_volatility(dates=date_range, expiry=t1,
                                                      delta="50", put_call="CALL")
                iv_far = bq.data.implied_volatility(dates=date_range, expiry=t2,
                                                     delta="50", put_call="CALL")
            else:
                moneyness = strike_sel.split("%")[0]
                iv_near = bq.data.implied_volatility(dates=date_range, expiry=t1,
                                                      pct_moneyness=moneyness)
                iv_far = bq.data.implied_volatility(dates=date_range, expiry=t2,
                                                     pct_moneyness=moneyness)

            req = bql.Request(ticker, {"IV_NEAR": iv_near, "IV_FAR": iv_far})
            resp = safe_bql(req, "Forward factor")
            if resp is None:
                clear_output(wait=True)
                display(error_msg("Query failed."))
                return

            dfs = multi_resp_to_dfs(resp)
            if len(dfs) < 2:
                clear_output(wait=True)
                display(error_msg("Insufficient data."))
                return

            df_n = dfs[0].reset_index()
            df_f = dfs[1].reset_index()
            dc_n, vc_n = find_cols(df_n)
            dc_f, vc_f = find_cols(df_f)

            if not all([dc_n, vc_n, dc_f, vc_f]):
                clear_output(wait=True)
                display(error_msg("Could not parse columns."))
                return

            near = df_n.rename(columns={dc_n: "date", vc_n: "iv_near"}).dropna(subset=["iv_near"])
            far = df_f.rename(columns={dc_f: "date", vc_f: "iv_far"}).dropna(subset=["iv_far"])
            merged = pd.merge(near[["date", "iv_near"]], far[["date", "iv_far"]],
                              on="date", how="inner").sort_values("date")

            if merged["iv_near"].median() < 1:
                merged["iv_near"] *= 100
            if merged["iv_far"].median() < 1:
                merged["iv_far"] *= 100

            T1 = t1_days / 365.0
            T2 = t2_days / 365.0
            var_near = (merged["iv_near"] ** 2) * T1
            var_far = (merged["iv_far"] ** 2) * T2
            fwd_var = (var_far - var_near) / (T2 - T1)
            merged["fwd_vol"] = np.where(fwd_var > 0, np.sqrt(fwd_var), np.nan)
            merged["fwd_factor"] = (merged["iv_near"] / merged["fwd_vol"]) * 100
            merged["simple_ratio"] = (merged["iv_near"] / merged["iv_far"]) * 100

            clear_output(wait=True)

            fig = styled_fig(
                f"{idx} {strike_sel} — Forward Factor ({t1} / {t1}→{t2})",
                height=520, rows=2, cols=1,
                subplot_titles=[
                    f"Forward Factor: {t1} / Fwd({t1}→{t2}) (%)",
                    f"Simple Ratio: {t1} / {t2} (%)"],
                vertical_spacing=0.12)

            fig.add_trace(go.Scatter(x=merged["date"], y=merged["fwd_factor"], name="Fwd Factor",
                                      line=dict(color=COLORS["fwd_factor"], width=2)), row=1, col=1)
            fig.add_hline(y=100, line_dash="dot", line_color="white", opacity=0.4, row=1, col=1)
            fig.update_yaxes(title_text="%", row=1, col=1)

            fig.add_trace(go.Scatter(x=merged["date"], y=merged["simple_ratio"], name="Simple Ratio",
                                      line=dict(color=COLORS["ratio"], width=2)), row=2, col=1)
            fig.add_hline(y=100, line_dash="dot", line_color="white", opacity=0.4, row=2, col=1)
            fig.update_yaxes(title_text="%", row=2, col=1)

            display(go.FigureWidget(fig))

    refresh_btn.on_click(_refresh)
    tab = widgets.VBox([widgets.HBox([idx_dd, t1_dd, t2_dd, strike_dd, lb_dd, refresh_btn]), out])
    _refresh()
    return tab


In [ ]:
# ==========================================================================
# TAB 5 — VOL BY DELTA (IV SMILE / SKEW)
# ==========================================================================

def build_tab_vol_by_delta():
    out = widgets.Output()
    idx_dd = widgets.Dropdown(options=list(INDEX_MAP.keys()), value="SPX",
                               description="Index:", style={"description_width": "80px"})
    term_dd = widgets.Dropdown(options=EXPIRY_OPTIONS, value="30D",
                                description="Expiry:", style={"description_width": "80px"})
    refresh_btn = widgets.Button(description="Refresh", button_style="primary",
                                  layout=widgets.Layout(width="100px"))

    # Available deltas in BQL
    DELTAS = DELTA_OPTIONS  # ["10", "25", "40", "50", "60", "75", "90"]

    def _refresh(b=None):
        with out:
            clear_output(wait=True)
            display(status_msg("Loading vol smile by delta…"))

            idx = idx_dd.value
            ticker = INDEX_MAP[idx]
            expiry = term_dd.value

            # Query IV at each delta for both puts and calls
            fields = {}
            for d in DELTAS:
                fields[f"PUT_{d}"] = bq.data.implied_volatility(
                    expiry=expiry, delta=d, put_call="PUT")
                fields[f"CALL_{d}"] = bq.data.implied_volatility(
                    expiry=expiry, delta=d, put_call="CALL")

            req = bql.Request(ticker, fields)
            resp = safe_bql(req, "Vol by delta")
            if resp is None:
                clear_output(wait=True)
                display(error_msg("Query failed."))
                return

            dfs = multi_resp_to_dfs(resp)

            # Parse results: puts then calls
            put_vols = {}
            call_vols = {}
            for i, d in enumerate(DELTAS):
                # Put
                pi = i * 2
                if pi < len(dfs) and not dfs[pi].empty:
                    val = dfs[pi].iloc[0].values[0]
                    if val is not None and not (isinstance(val, float) and np.isnan(val)):
                        put_vols[int(d)] = float(val) * 100 if float(val) < 1 else float(val)
                # Call
                ci = i * 2 + 1
                if ci < len(dfs) and not dfs[ci].empty:
                    val = dfs[ci].iloc[0].values[0]
                    if val is not None and not (isinstance(val, float) and np.isnan(val)):
                        call_vols[int(d)] = float(val) * 100 if float(val) < 1 else float(val)

            clear_output(wait=True)

            # Build smile: OTM puts (high delta = deep ITM put = OTM call) + ATM + OTM calls
            # Convention: put delta 90 = deep OTM put, put delta 10 = deep ITM put
            # Plot x-axis as "effective delta" from put side
            put_deltas = sorted(put_vols.keys())
            call_deltas = sorted(call_vols.keys())

            fig = styled_fig(f"{idx} — {expiry} IV Smile", height=450)

            if put_vols:
                fig.add_trace(go.Scatter(
                    x=put_deltas, y=[put_vols[d] for d in put_deltas],
                    mode="lines+markers", name="Put IV",
                    marker=dict(size=9, color=COLORS["rv"]),
                    line=dict(color=COLORS["rv"], width=2),
                    hovertemplate="Put Δ%{x}: %{y:.1f}%<extra></extra>"))

            if call_vols:
                fig.add_trace(go.Scatter(
                    x=call_deltas, y=[call_vols[d] for d in call_deltas],
                    mode="lines+markers", name="Call IV",
                    marker=dict(size=9, color=COLORS["iv"]),
                    line=dict(color=COLORS["iv"], width=2),
                    hovertemplate="Call Δ%{x}: %{y:.1f}%<extra></extra>"))

            fig.update_xaxes(title_text="Delta", dtick=5)
            fig.update_yaxes(title_text="IV (%)")

            # ATM annotation
            if 50 in put_vols:
                fig.add_annotation(x=50, y=put_vols[50], text=f"ATM: {put_vols[50]:.1f}%",
                                    showarrow=True, arrowhead=2, arrowcolor="white",
                                    font=dict(color="white", size=11),
                                    bgcolor="rgba(30,30,47,0.8)")

            # 25Δ skew
            if 25 in put_vols and 25 in call_vols:
                skew = put_vols[25] - call_vols[25]
                fig.add_annotation(x=0.99, y=0.01, xref="paper", yref="paper",
                                    text=f"25Δ Put-Call Skew: {skew:+.1f}",
                                    showarrow=False, font=dict(size=13, color=COLORS["fwd_factor"]),
                                    bgcolor="rgba(30,30,47,0.8)", borderpad=6)

            display(go.FigureWidget(fig))

    refresh_btn.on_click(_refresh)
    tab = widgets.VBox([widgets.HBox([idx_dd, term_dd, refresh_btn]), out])
    _refresh()
    return tab


In [ ]:
# ==========================================================================
# TAB 6 — VOL SURFACE HEATMAP (5Y PERCENTILE)
# ==========================================================================

def build_tab_vol_heatmap():
    out = widgets.Output()
    idx_dd = widgets.Dropdown(options=list(INDEX_MAP.keys()), value="SPX",
                               description="Index:", style={"description_width": "80px"})
    refresh_btn = widgets.Button(description="Refresh", button_style="primary",
                                  layout=widgets.Layout(width="100px"))

    # Grid matching BQL available values
    TENORS = ["30D", "60D", "90D", "180D", "360D", "720D"]
    TENOR_LABELS = ["1M", "2M", "3M", "6M", "1Y", "2Y"]
    STRIKES = ["80", "90", "95", "97", "100", "102", "105", "110", "120"]

    def _refresh(b=None):
        with out:
            clear_output(wait=True)
            display(status_msg(
                "Building vol surface heatmap (5Y percentiles)…<br>"
                "Querying tenor×strike grid — this may take 30-60 seconds."))

            idx = idx_dd.value
            ticker = INDEX_MAP[idx]
            date_range = bq.func.range("-5Y", "0D")

            all_data = {}

            for ti, tenor in enumerate(TENORS):
                # Current IV at each strike for this tenor
                fields_cur = {}
                fields_hist = {}
                for strike in STRIKES:
                    key = f"S{strike}"
                    fields_cur[key] = bq.data.implied_volatility(
                        expiry=tenor, pct_moneyness=strike)
                    fields_hist[key] = bq.data.implied_volatility(
                        expiry=tenor, pct_moneyness=strike, dates=date_range)

                resp_cur = safe_bql(
                    bql.Request(ticker, fields_cur), f"Heatmap cur {TENOR_LABELS[ti]}")
                resp_hist = safe_bql(
                    bql.Request(ticker, fields_hist), f"Heatmap hist {TENOR_LABELS[ti]}")

                if resp_cur is None or resp_hist is None:
                    continue

                dfs_cur = multi_resp_to_dfs(resp_cur)
                dfs_hist = multi_resp_to_dfs(resp_hist)

                for si, strike in enumerate(STRIKES):
                    try:
                        cur_val = np.nan
                        if si < len(dfs_cur) and not dfs_cur[si].empty:
                            # Get value — skip DATE/CURRENCY cols
                            row = dfs_cur[si].iloc[0]
                            val_cols = [c for c in dfs_cur[si].columns
                                        if "DATE" not in c.upper() and "CURRENCY" not in c.upper()]
                            if val_cols:
                                cur_val = float(row[val_cols[0]])

                        hist_vals = np.array([])
                        if si < len(dfs_hist) and not dfs_hist[si].empty:
                            val_cols = [c for c in dfs_hist[si].columns
                                        if "DATE" not in c.upper() and "CURRENCY" not in c.upper()
                                        and "ID" not in c.upper()]
                            if val_cols:
                                hist_vals = dfs_hist[si][val_cols[0]].dropna().values.astype(float)

                        if not np.isnan(cur_val) and len(hist_vals) > 20:
                            pctile = (hist_vals < cur_val).sum() / len(hist_vals)
                            all_data[(ti, strike)] = round(pctile, 2)
                        else:
                            all_data[(ti, strike)] = np.nan
                    except Exception:
                        all_data[(ti, strike)] = np.nan

            clear_output(wait=True)

            # Build matrix
            z = []
            text_vals = []
            for ti in range(len(TENORS)):
                row = []
                text_row = []
                for strike in STRIKES:
                    val = all_data.get((ti, strike), np.nan)
                    row.append(val)
                    text_row.append(f"{val:.2f}" if not np.isnan(val) else "")
                z.append(row)
                text_vals.append(text_row)

            fig = styled_fig(f"{idx} — Implied Vol 5Y Percentile Heatmap", height=480)
            fig.add_trace(go.Heatmap(
                z=z,
                x=[f"{s}%" for s in STRIKES],
                y=TENOR_LABELS,
                text=text_vals,
                texttemplate="%{text}",
                textfont=dict(size=12, color="black"),
                colorscale=[
                    [0.0, COLORS["heatmap_low"]],
                    [0.5, COLORS["heatmap_mid"]],
                    [1.0, COLORS["heatmap_high"]]],
                zmin=0, zmax=1,
                colorbar=dict(title="Pctile", tickformat=".0%"),
                hovertemplate="Tenor: %{y}<br>Strike: %{x}<br>Pctile: %{z:.0%}<extra></extra>"))

            fig.update_xaxes(title_text="Strike (% of Spot)", side="top")
            fig.update_yaxes(title_text="Tenor", autorange="reversed")
            fig.update_layout(margin=dict(t=80))
            display(go.FigureWidget(fig))

            # Skew table
            skew_rows = []
            for ti in range(len(TENORS)):
                p90 = all_data.get((ti, "90"), np.nan)
                p100 = all_data.get((ti, "100"), np.nan)
                p110 = all_data.get((ti, "110"), np.nan)
                def _d(a, b):
                    return f"{a-b:.2f}" if not (np.isnan(a) or np.isnan(b)) else ""
                skew_rows.append({
                    "Tenor": TENOR_LABELS[ti],
                    "90-100": _d(p90, p100),
                    "100-110": _d(p100, p110),
                    "90-110": _d(p90, p110)})

            display(HTML(
                "<div style='color:#E0E0E0; font-family:Consolas,monospace; margin-top:12px;'>"
                "<b>Skew Percentile Differentials</b></div>"))
            display(HTML(pd.DataFrame(skew_rows).to_html(index=False, classes="table",
                                                          border=0, justify="center")))

    refresh_btn.on_click(_refresh)
    controls = widgets.HBox([idx_dd, refresh_btn])
    # Don't auto-load this tab — it's expensive
    tab = widgets.VBox([
        controls,
        widgets.HTML("<p style='color:#8899AA; font-family:Consolas,monospace; font-size:12px; margin:4px 0;'>"
             "Click Refresh to load. This tab queries ~54 tenor×strike combos with 5Y history.</p>"),
        out])
    return tab



In [ ]:
# ==========================================================================
# DASHBOARD ASSEMBLY
# ==========================================================================

display(HTML("""
<style>
    .widget-tab > .p-TabBar .p-TabBar-tab {
        font-family: Consolas, monospace;
        font-size: 13px;
        min-width: 130px;
    }
    table.table {
        color: #E0E0E0; font-family: Consolas, monospace; font-size: 12px;
        border-collapse: collapse; margin-top: 4px;
    }
    table.table th, table.table td {
        padding: 4px 10px; border: 1px solid #444; text-align: center;
    }
    table.table th { background: #2a2a3e; }
</style>
<div style="background:linear-gradient(90deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
            padding:16px 24px; border-radius:8px; margin-bottom:12px;">
    <h1 style="color:#E0E0E0; margin:0; font-family:Consolas,monospace; font-size:22px;">
        ⚡ Vol Analysis Dashboard
    </h1>
    <p style="color:#8899AA; margin:4px 0 0 0; font-size:13px; font-family:Consolas,monospace;">
        Quick-look volatility analytics · SPX · NDX · INDU · RTY
    </p>
</div>
"""))

tab_widget = widgets.Tab()
tab_widget.children = [
    build_tab_vix_term_structure(),
    build_tab_rv_iv(),
    build_tab_forward_vol(),
    build_tab_forward_factor(),
    build_tab_vol_by_delta(),
    build_tab_vol_heatmap(),
]

tab_widget.set_title(0, "VIX Term Structure")
tab_widget.set_title(1, "RV vs IV")
tab_widget.set_title(2, "Forward Vol")
tab_widget.set_title(3, "Forward Factor")
tab_widget.set_title(4, "Vol by Delta")
tab_widget.set_title(5, "Vol Heatmap")

display(tab_widget)

